# Turkish Legal RAG - Stable Colab Demo

This notebook is designed for the current Google Colab T4 runtime. It deliberately avoids `bitsandbytes`, LoRA, reranking, and CUDA-specific extension loading.

1. Select **Runtime > Change runtime type > T4 GPU**.
2. Run all cells.
3. Upload `kaggle.json` when prompted.
4. Open the `gradio.live` URL printed by the final cell.

Expected startup time: approximately 5-10 minutes.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU secin.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA runtime:', torch.version.cuda)

## 2. Install minimal dependencies

In [ ]:
!pip install -q "gradio>=4.44,<6" "transformers==4.45.2" "accelerate>=0.33,<2" "rank-bm25==0.2.2" "kaggle>=1.6"
print('Dependencies installed. No bitsandbytes is used.')

## 3. Upload Kaggle API token

In [ ]:
from google.colab import files
from pathlib import Path
import os

uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'kaggle.json yuklenmedi.'
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
token_path = kaggle_dir / 'kaggle.json'
token_path.write_bytes(uploaded['kaggle.json'])
os.chmod(token_path, 0o600)
print('Kaggle credentials ready.')

## 4. Download the index and latest application code

In [ ]:
!rm -rf /content/NLP-RAG /content/legal-rag-artifacts
!git clone -q --depth 1 https://github.com/gokaycetinn/NLP-RAG.git /content/NLP-RAG
!mkdir -p /content/legal-rag-artifacts
!kaggle datasets download -q -d hasanemreusta/turkish-legal-rag-system -p /content/legal-rag-artifacts --unzip
print('Repository and artifacts downloaded.')

## 5. Connect the BM25 index

In [ ]:
from pathlib import Path
import os

repo = Path('/content/NLP-RAG')
artifact_root = Path('/content/legal-rag-artifacts')
bm25_files = list(artifact_root.rglob('bm25.pkl'))
index_dir = next((p.parent for p in bm25_files if (p.parent / 'meta.jsonl').exists()), None)
assert index_dir is not None, 'BM25 index bulunamadi.'
target = repo / 'data' / 'index_full'
target.parent.mkdir(parents=True, exist_ok=True)
if target.is_symlink() or target.is_file():
    target.unlink()
elif target.exists():
    import shutil
    shutil.rmtree(target)
target.symlink_to(index_dir, target_is_directory=True)
os.chdir(repo)
print('BM25 index:', index_dir)
print('Ready:', Path('data/index_full/bm25.pkl').exists())

## 6. Launch the app

The first launch downloads the approximately 3 GB FP16 generator. Open the public `gradio.live` URL when it appears.

In [ ]:
%cd /content/NLP-RAG
!python -m src.demo.colab_lite_app